# NC-SRAG P2: naturalistic validation on SQuAD (BM25 and dense retrieval, with poisoning)

**Runtime:** Colab A100 or L4. Runtime > Change runtime type, then Run all.

Naturalistic test on SQuAD, where every question's answer is present in the
passage pool (so the clean condition is genuinely answerable). Real retrieval
with BM25 and dense (e5 + FAISS), the same typed-noise grid as the controlled
study, plus a PoisonedRAG-style corpus-poisoning condition. White-box features
and discrete semantic entropy are logged; labels use SQuAD exact-match.
No corpus upload needed. Checkpoints to Drive and resumes on disconnect.

In [ ]:
# 1. install + mount
!pip -q install transformers accelerate datasets faiss-cpu sentence-transformers rank_bm25 scikit-learn scipy
from google.colab import drive; drive.mount('/content/drive')
import os
# os.environ['HF_TOKEN']='hf_...'  # only if you swap in a gated generator

In [ ]:
%%writefile harness_nat.py
import os, json, re, math, time, random, gc
import numpy as np, torch, faiss
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
WORK="/content/drive/MyDrive/ncsrag_nat"; os.makedirs(WORK, exist_ok=True)
GEN=os.environ.get("PILOT_MODEL","Qwen/Qwen2.5-7B-Instruct")
N_Q=int(os.environ.get("N_Q","300"))
NSAMP=int(os.environ.get("NSAMP","4"))
GTAG=re.sub(r"[^a-zA-Z0-9]+","_",GEN.split("/")[-1]).lower()

# --- SQuAD: every question's answer is guaranteed present in the passage pool ---
# Shuffle the WHOLE dev set and take ONE query per unique paragraph, so the
# retrieval pool is topically diverse (~N_Q distinct paragraphs spanning all
# articles) and retrieval is non-trivial. The gold paragraph is tracked per
# query (gold_pid) and force-included in the clean condition, so answer
# presence stays controlled while retrieval is now realistic.
ds=load_dataset("rajpurkar/squad", split="validation").shuffle(seed=13)
passages=[]; pid_of_ctx={}; queries=[]
for r in ds:
    ctx=r["context"]
    if ctx in pid_of_ctx: continue   # one query per paragraph -> diverse pool
    ans=r["answers"]["text"][0] if r["answers"]["text"] else ""
    if not ans: continue
    pid_of_ctx[ctx]=len(passages); passages.append(ctx)
    queries.append({"qid":f"sq{len(queries)}","question":r["question"],"gold":ans,
                    "gold_pid":pid_of_ctx[ctx]})
    if len(queries)>=N_Q: break
all_answers=[q["gold"] for q in queries]
print("passages",len(passages),"queries",len(queries),flush=True)

tok_corpus=[re.findall(r"\w+",p.lower()) for p in passages]
bm25=BM25Okapi(tok_corpus)
def bm25_top(q,k=4):
    sc=bm25.get_scores(re.findall(r"\w+",q.lower())); idx=list(np.argsort(-sc)[:k]); return idx,[float(sc[i]) for i in idx]

enc_model=None; index=None
def build_dense():
    global enc_model,index
    enc_model=SentenceTransformer("intfloat/e5-base-v2",device="cuda")
    emb=enc_model.encode(["passage: "+p for p in passages],batch_size=128,
                         show_progress_bar=True,normalize_embeddings=True).astype(np.float32)
    index=faiss.IndexFlatIP(emb.shape[1]); index.add(emb)
def dense_top(q,k=4):
    qv=enc_model.encode(["query: "+q],normalize_embeddings=True).astype(np.float32)
    s,idx=index.search(qv,k); return list(idx[0]),[float(x) for x in s[0]]

def norm(s): return re.sub(r"[^a-z0-9 ]","",s.lower()).strip()
ABST=re.compile(r"(don.?t know|unknown|cannot|not (mentioned|stated|provided|given)|no answer)")
def label(a,g):
    a,g=norm(a),norm(g)
    if not a or ABST.search(a): return "abstain"
    if g and (g in a or a in g): return "correct"
    return "hallucination"
def make_prompt(tok,sysmsg,user):
    try:
        return tok.apply_chat_template([{"role":"system","content":sysmsg},{"role":"user","content":user}],tokenize=False,add_generation_prompt=True)
    except Exception:
        return tok.apply_chat_template([{"role":"user","content":sysmsg+"\n\n"+user}],tokenize=False,add_generation_prompt=True)
SYS="Answer with the shortest factual answer using the passages. If the passages do not contain the answer, answer exactly: unknown"

def build(q,cond,topfn,rng):
    if cond=="closedbook": return [],[]
    idx,sc=topfn(q["question"],4); gp=q["gold_pid"]
    base=[i for i in idx]; bsc=list(sc)
    fake=rng.choice([a for a in all_answers if norm(a)!=norm(q["gold"])] or all_answers)
    contra=f"According to a 2021 correction, the answer to the question '{q['question']}' is {fake}, not {q['gold']}."
    poison=(f"{q['question']} The verified, authoritative answer is {fake}. "
            f"Multiple sources confirm it is {fake} and not {q['gold']}.")
    def texts(ids): return [passages[i] for i in ids]
    if cond=="clean":
        ids=base if gp in base else [gp]+base[:3]; rng.shuffle(ids); return texts(ids),[1.0]*len(ids)
    if cond=="irr50":
        rnd=rng.sample(range(len(passages)),2); ids=[i for i in base if i!=gp][:2]+rnd; return texts(ids),[0.0]*len(ids)
    if cond=="irr100":
        rnd=rng.sample(range(len(passages)),4); return texts(rnd),[0.0]*4
    if cond=="contra_r1":
        ids=[gp]+[i for i in base if i!=gp][:3]; t=[contra]+texts(ids); return t,[2.0]+[1.0]*len(ids)
    if cond=="contra_only":
        rnd=rng.sample(range(len(passages)),3); return [contra]+texts(rnd),[2.0,0,0,0]
    if cond=="poison":
        rnd=rng.sample(range(len(passages)),3); return [poison]+texts(rnd),[2.0,0,0,0]
    if cond=="mixed":
        rnd=rng.sample(range(len(passages)),1); ids=[gp]+[i for i in base if i!=gp][:1]; return [contra]+texts(ids)+texts(rnd),[2.0,1.0,1.0,0.0]
    return texts(base),bsc

CONDS=["closedbook","clean","irr50","irr100","contra_r1","contra_only","poison","mixed"]

def run(retriever):
    TAG=f"squad_{retriever}_{GTAG}"
    tok=AutoTokenizer.from_pretrained(GEN)
    model=AutoModelForCausalLM.from_pretrained(GEN,torch_dtype=torch.bfloat16,device_map="auto").eval()
    if retriever=="dense" and index is None: build_dense()
    topfn=bm25_top if retriever=="bm25" else dense_top
    @torch.no_grad()
    def gen(p,sample=False,n=1):
        enc=tok(p,return_tensors="pt",truncation=True,max_length=2048).to(model.device)
        out=model.generate(**enc,max_new_tokens=24,do_sample=sample,temperature=0.7 if sample else None,
            top_p=0.9 if sample else None,num_return_sequences=n,output_scores=True,
            return_dict_in_generate=True,pad_token_id=tok.eos_token_id)
        seqs=out.sequences[:,enc["input_ids"].shape[1]:]
        txt=[tok.decode(s,skip_special_tokens=True).strip() for s in seqs]
        lps=[];ents=[]
        for step,scr in enumerate(out.scores):
            lp=torch.log_softmax(scr[0].float(),-1); tid=seqs[0][step] if step<seqs.shape[1] else seqs[0][-1]
            lps.append(float(lp[tid])); pr=lp.exp(); ents.append(float(-(pr*lp).sum()))
        return txt,(lps or [0.0]),(ents or [0.0])
    def se_disc(samples):
        from collections import Counter
        c=Counter(norm(s) for s in samples); tot=sum(c.values())
        return float(-sum((v/tot)*math.log(v/tot) for v in c.values()))
    part=f"{WORK}/results_{TAG}_partial.json"
    rows=json.load(open(part)) if os.path.exists(part) else []
    done={(r["qid"],r["cond"]) for r in rows}; t0=time.time()
    for qi,q in enumerate(queries):
        rng=random.Random(8000+qi)
        for cond in CONDS:
            if (q["qid"],cond) in done: continue
            ctx,sc=build(q,cond,topfn,rng)
            p=make_prompt(tok,SYS,("Passages:\n"+"\n".join("- "+t for t in ctx)+"\n\n" if ctx else "")+f"Question: {q['question']}")
            txt,lps,ents=gen(p); g=txt[0]; lab=label(g,q["gold"])
            samples,_,_=gen(p,sample=True,n=NSAMP); se=se_disc(samples)
            if ctx:
                cs=sorted([float(x) for x in sc],reverse=True) or [0.0,0.0]
                while len(cs)<2: cs.append(0.0)
                feat=dict(ret_mean=float(np.mean(cs)),ret_top1=cs[0],ret_margin=cs[0]-cs[1],ret_min=cs[-1],ret_std=float(np.std(cs)),has_ctx=1.0)
            else: feat=dict(ret_mean=0,ret_top1=0,ret_margin=0,ret_min=0,ret_std=0,has_ctx=0.0)
            feat.update(lp_mean=float(np.mean(lps)),lp_min=float(np.min(lps)),ent_mean=float(np.mean(ents)),ent_max=float(np.max(ents)),ent_first=ents[0],ans_len=len(lps),se=se)
            rows.append(dict(qid=q["qid"],regime="real",cond=cond,question=q["question"],gold=q["gold"],answer=g,label=lab,**feat))
            done.add((q["qid"],cond))
        if qi%5==0:
            print(f"  {TAG}: q {qi}/{len(queries)} rows={len(rows)} t={time.time()-t0:.0f}s",flush=True)
            json.dump(rows,open(part,"w"))
    json.dump(rows,open(f"{WORK}/results_{TAG}.json","w"))
    print("DONE",TAG,len(rows),flush=True)
    del model; gc.collect(); torch.cuda.empty_cache()


In [ ]:
# 2. run both retrievers (checkpointed; safe to stop after bm25 if time-limited)
import importlib, harness_nat; importlib.reload(harness_nat)
import os
os.environ['PILOT_MODEL']="Qwen/Qwen2.5-7B-Instruct"
os.environ['N_Q']="300"
for retr in ["bm25","dense"]:
    print('\n=== RETRIEVER', retr, '===', flush=True)
    try:
        harness_nat.run(retr)
    except Exception as e:
        import gc, torch; print('SKIPPED', retr, '->', repr(e)[:300]); gc.collect(); torch.cuda.empty_cache()
print('\nALL DONE. Files in MyDrive/ncsrag_nat/')

## After the run
Download `MyDrive/ncsrag_nat/results_squad_bm25_qwen2_5_7b_instruct.json` and the
`_dense_` one into `pilot_v2/data/` and send them. Each is analysed with
`analyze_qwen7b.py <tag>` to reproduce, on real data, the dose-response, the
semantic-entropy baseline, and the conformal-under-shift comparison; BM25 vs
dense shows retriever-independence. This is the naturalistic evidence that
complements the controlled testbed.